# Step 7: Staged Fine-tuning

**Goal:** Test whether sequential fine-tuning beats simultaneous mixing.

Your `mixed_50` model trained on 1350 arXiv + 1350 MARCO simultaneously and failed to beat the baseline. This runs the same data sequentially in both orders:

- `staged_gen_dom` — MARCO first (1350 pairs), then arXiv (1350 pairs)
- `staged_dom_gen` — arXiv first (1350 pairs), then MARCO (1350 pairs)

**Everything held constant:**
- Base model: `bge-base-en-v1.5` (same as step 2)
- Total pairs: 2700 (same as all previous runs)
- Loss: `MultipleNegativesRankingLoss`
- Epochs per stage: 3 (6 total — keeps total training steps comparable)
- Batch: 32, Warmup: 100, Seed: 42

**Direct comparison:** `staged_gen_dom` vs `staged_dom_gen` vs `mixed_50` (already done)

## 0. Runtime check

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())
if 'T4' not in result.stdout:
    print('WARNING: Not on T4. Go to Runtime > Change runtime type > T4 GPU')
else:
    print('Good to go.')

## 1. Install dependencies

In [ ]:
%%capture
!pip install sentence-transformers faiss-cpu accelerate

## 2. Upload files

Upload these three files when the dialog appears:
- `data/training_pairs.json`
- `data/marco_pairs.json`
- `experiments/baseline/baseline_chunks.pkl`

In [ ]:
from google.colab import files
import os

os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/experiments/baseline', exist_ok=True)

print('Upload: training_pairs.json, marco_pairs.json, baseline_chunks.pkl')
uploaded = files.upload()

dest_map = {
    'training_pairs.json':  '/content/data/training_pairs.json',
    'marco_pairs.json':     '/content/data/marco_pairs.json',
    'baseline_chunks.pkl':  '/content/experiments/baseline/baseline_chunks.pkl',
}
for fname, data in uploaded.items():
    dest = dest_map.get(fname, f'/content/{fname}')
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'Saved {fname} -> {dest}')

## 3. Config

In [ ]:
import random
import numpy as np

BASE_MODEL      = 'BAAI/bge-base-en-v1.5'
STAGE_SIZE      = 1350   # pairs per stage (1350 + 1350 = 2700 total)
EPOCHS_PER_STAGE = 3     # 3 epochs x 2 stages = 6 total (comparable to 5 epochs on full set)
BATCH_SIZE      = 32
WARMUP_STEPS    = 100
SEED            = 42
EXPERIMENTS_DIR = '/content/experiments'

random.seed(SEED)
np.random.seed(SEED)

print(f'Base model       : {BASE_MODEL}')
print(f'Pairs per stage  : {STAGE_SIZE}')
print(f'Epochs per stage : {EPOCHS_PER_STAGE}')
print(f'Total pairs      : {STAGE_SIZE * 2}')

## 4. Load data

In [ ]:
import json

with open('/content/data/training_pairs.json') as f:
    all_arxiv = json.load(f)
with open('/content/data/marco_pairs.json') as f:
    all_marco = json.load(f)

# Same split as all previous runs — never touch last 100
train_pool  = all_arxiv[:-100]
val_pairs   = train_pool[2700:]    # pairs 2700-2900 = validation

# Stage data — sample with fixed seed for reproducibility
rng = random.Random(SEED)
arxiv_stage = rng.sample(train_pool[:2700], STAGE_SIZE)   # 1350 arXiv
marco_stage = rng.sample(all_marco,         STAGE_SIZE)   # 1350 MARCO

print(f'arXiv stage pairs : {len(arxiv_stage)}')
print(f'MARCO stage pairs : {len(marco_stage)}')
print(f'Validation pairs  : {len(val_pairs)}')

## 5. Validation evaluator

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

def make_evaluator(val_pairs, name):
    queries  = {str(i): p['query']    for i, p in enumerate(val_pairs)}
    corpus   = {str(i): p['positive'] for i, p in enumerate(val_pairs)}
    relevant = {str(i): {str(i)}      for i in range(len(val_pairs))}
    return InformationRetrievalEvaluator(queries, corpus, relevant, name=name)

evaluator = make_evaluator(val_pairs, name='val-staged')
print(f'Evaluator ready — {len(val_pairs)} val pairs')

## 6. Training function

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader

def train_stage(model, pairs, stage_name, evaluator, out_dir):
    """Train one stage. Accepts an existing model so stages chain together."""
    examples = [InputExample(texts=[p['query'], p['positive']]) for p in pairs]
    loader   = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
    loss     = MultipleNegativesRankingLoss(model)

    print(f'\n  Stage: {stage_name} — {len(pairs)} pairs, {EPOCHS_PER_STAGE} epochs')

    model.fit(
        train_objectives=[(loader, loss)],
        evaluator=evaluator,
        epochs=EPOCHS_PER_STAGE,
        warmup_steps=WARMUP_STEPS,
        evaluation_steps=len(loader),
        output_path=out_dir,
        save_best_model=True,
        show_progress_bar=True,
        use_amp=True
    )
    # Reload best saved model before next stage
    return SentenceTransformer(out_dir)

## 7a. Run 1 — General → Domain (MARCO first, then arXiv) (~20 min)

In [ ]:
import os

run_name = 'staged_gen_dom'
out_dir  = os.path.join(EXPERIMENTS_DIR, run_name, 'model')
os.makedirs(out_dir, exist_ok=True)

print(f'=== {run_name}: MARCO → arXiv ===')
model = SentenceTransformer(BASE_MODEL)

# Stage 1: general (MARCO)
model = train_stage(model, marco_stage, 'MARCO (general)', evaluator, out_dir)

# Stage 2: domain (arXiv) — continues from stage 1 weights
model = train_stage(model, arxiv_stage, 'arXiv (domain)', evaluator, out_dir)

print(f'\nSaved to {out_dir}')

## 7b. Run 2 — Domain → General (arXiv first, then MARCO) (~20 min)

In [ ]:
run_name = 'staged_dom_gen'
out_dir  = os.path.join(EXPERIMENTS_DIR, run_name, 'model')
os.makedirs(out_dir, exist_ok=True)

print(f'=== {run_name}: arXiv → MARCO ===')
model = SentenceTransformer(BASE_MODEL)

# Stage 1: domain (arXiv)
model = train_stage(model, arxiv_stage, 'arXiv (domain)', evaluator, out_dir)

# Stage 2: general (MARCO) — continues from stage 1 weights
model = train_stage(model, marco_stage, 'MARCO (general)', evaluator, out_dir)

print(f'\nSaved to {out_dir}')

## 8. Build FAISS indexes for both models

In [ ]:
import pickle
import faiss
import numpy as np

with open('/content/experiments/baseline/baseline_chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)
print(f'Loaded {len(chunks)} chunks')

def build_index(model_dir, run_name):
    print(f'\nBuilding index for {run_name}...')
    model      = SentenceTransformer(model_dir)
    texts      = [c['text'] for c in chunks]
    embeddings = model.encode(texts, batch_size=256, show_progress_bar=True,
                               normalize_embeddings=True)
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    idx_path = os.path.join(EXPERIMENTS_DIR, run_name, f'{run_name}.faiss')
    faiss.write_index(index, idx_path)
    print(f'Saved: {index.ntotal} vectors, dim={dim}')

build_index(os.path.join(EXPERIMENTS_DIR, 'staged_gen_dom', 'model'), 'staged_gen_dom')
build_index(os.path.join(EXPERIMENTS_DIR, 'staged_dom_gen', 'model'), 'staged_dom_gen')

## 9. Download both models

In [ ]:
import shutil
from google.colab import files

for run_name in ['staged_gen_dom', 'staged_dom_gen']:
    zip_path = f'/content/{run_name}'
    shutil.make_archive(zip_path, 'zip', EXPERIMENTS_DIR, run_name)
    files.download(f'{zip_path}.zip')
    print(f'Downloading {run_name}.zip...')

print('\nUnzip both into your experiments/ folder on Mac.')
print('Then add both to BGE_RUNS in step4d_eval_bge.py and run:')
print('  python step4d_eval_bge.py --eval manual')